In [10]:
import random
import hashlib
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes


def generate_random_bits(length):
    """Generate a list of random bits (0 or 1)."""
    return [random.randint(0, 1) for _ in range(length)]


def generate_random_bases(length):
    """Generate a list of random bases ('+' or 'x')."""
    return [random.choice(['+', 'x']) for _ in range(length)]


def encode_qubits(bits, bases):
    """Encode bits into qubits using the chosen bases."""
    return [(bit, base) for bit, base in zip(bits, bases)]


def measure_qubits(qubits, bases):
    """Measure qubits based on the receiver's bases."""
    measured_bits = []
    for (bit, base), recv_base in zip(qubits, bases):
        if base == recv_base:
            measured_bits.append(bit)
        else:
            measured_bits.append(random.randint(0, 1))
    return measured_bits


def compare_bases(sender_bases, receiver_bases):
    """Compare the sender's and receiver's bases to find matching indices."""
    return [i for i in range(len(sender_bases)) if sender_bases[i] == receiver_bases[i]]


def generate_key(bits, indices):
    """Generate the final key using bits at matching indices."""
    res= [bits[i] for i in indices]
    
   
    return res


def bits_to_bytes(bits):
    """Convert a list of bits into a bytes object."""
    byte_array = bytearray()
    for i in range(0, len(bits), 8):
        byte_chunk = bits[i:i+8]
        byte_array.append(int(''.join(map(str, byte_chunk)), 2))
    return bytes(byte_array)


def derive_aes_key(shared_key_bits):
    """
    Derive an AES key from the shared key bits.
    Convert the bits to bytes and hash them to get a 256-bit AES key.
    """
    shared_key_bytes = bits_to_bytes(shared_key_bits)
    return hashlib.sha256(shared_key_bytes).digest()  # Derive a 256-bit AES key


def aes_encrypt(message, key):
    """
    Encrypt a message using AES encryption.
    """
   
    cipher = AES.new(key, AES.MODE_CBC)  # Using CBC mode
    ciphertext = cipher.encrypt(pad(message.encode('utf-8'), AES.block_size))
    encrytion_time = time.time() 
    return cipher.iv, ciphertext


def aes_decrypt(iv, ciphertext, key):
    """
    Decrypt a ciphertext using AES decryption.
    """
    start_time = time.time()
    cipher = AES.new(key, AES.MODE_CBC, iv)
    plaintext = unpad(cipher.decrypt(ciphertext), AES.block_size)
    decrption_time = time.time() 
    return plaintext.decode('utf-8')

def save_to_file(filename, data):
    """Save binary data to a file."""
    with open(filename, 'wb') as file:
        file.write(data)


def load_from_file(filename):
    """Load binary data from a file."""
    with open(filename, 'rb') as file:
        return file.read()


# Simulation parameters
key_length = 128  # Length of the raw key

# Step 1: Sender generates random bits and bases
sender_bits = generate_random_bits(key_length)
sender_bases = generate_random_bases(key_length)

# Step 2: Receiver generates random bases
receiver_bases = generate_random_bases(key_length)

# Step 3: Sender encodes qubits and receiver measures them
qubits = encode_qubits(sender_bits, sender_bases)
receiver_measured_bits = measure_qubits(qubits, receiver_bases)

# Step 4: Compare bases and generate the shared key
matching_indices = compare_bases(sender_bases, receiver_bases)
sender_key = generate_key(sender_bits, matching_indices)
receiver_key = generate_key(receiver_measured_bits, matching_indices)

# Ensure keys match (for simulation purposes)
assert sender_key == receiver_key, "Keys do not match! Simulation error."

# Step 5: Derive AES key from the shared QKD key
aes_key = derive_aes_key(sender_key)

# Step 6: Message encryption using AES
original_message = "HELLO QUANTUM WORLD"  # Message to encrypt
print("Original Message:       ", original_message)

iv, ciphertext = aes_encrypt(original_message, aes_key)
print("Ciphertext:             ", ciphertext)
save_to_file("iv.bin", iv)
save_to_file("ciphertext.bin", ciphertext)
loaded_iv = load_from_file("iv.bin")
loaded_ciphertext = load_from_file("ciphertext.bin")


# Step 7: Message decryption using AES
aes_key = derive_aes_key(sender_key)
decrypted_message = aes_decrypt(loaded_iv, loaded_ciphertext, aes_key)

print("Decrypted Message:      ", decrypted_message)


# Step 9: Simulate eavesdropping detection
def detect_eavesdropping(bits_sender, bits_receiver):
    """Compare keys at random indices to detect eavesdropping."""
    return bits_sender != bits_receiver


eavesdropping_detected = detect_eavesdropping(sender_key, receiver_key)
print("Eavesdropping detected: ", eavesdropping_detected)



Original Message:        HELLO QUANTUM WORLD


NameError: name 'time' is not defined